# Jupyter 

In [2]:
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
import random
import tensorflow as tf
import numpy as np
import timeit
from sklearn.neighbors import NearestNeighbors
from sklearn import metrics
import scipy.io as sio
import csv
import os
import math 
import random as rand
from numpy import genfromtxt
from numpy import sum,isrealobj,sqrt
from numpy.random import standard_normal

from Modules.Evaluations import ConfidenceEstimation
from Modules.Graphs import ROC

from scipy.signal.signaltools import wiener
from sklearn.neighbors import KNeighborsRegressor
import warnings
from scipy.signal import savgol_filter
from scipy.spatial import distance
from fastdtw import fastdtw

from sklearn.preprocessing import MinMaxScaler
from IO import Writer

from Modules.Tools import MapTool
from Modules.Tools import BoxPlot

ModuleNotFoundError: No module named 'Evaluations'

In [ ]:
def NormalizeData(data, Min, Max):
    return (data - Min) / (Max - Min)

In [ ]:
# normalizes the whole dataset
def normalize(dataset):
    maxi = np.max(dataset)
    mini = np.min(dataset)
    N_dataset = np.copy(dataset)
    for s in range(dataset.shape[0]):
        for f in range(dataset.shape[1]):
            N_dataset[s][f] = (dataset[s][f] - mini)/(maxi - mini)
    return N_dataset

In [ ]:
def maximum(dataset):
    maxi = 0
    for s in range(dataset.shape[0]):
        for f in range(len(dataset[s])):
            if dataset[s][f] > maxi:
                maxi = dataset[s][f]
    
    return maxi

In [ ]:
def minimum(dataset):
    mini = 100000000000000
    for s in range(dataset.shape[0]):
        for f in range(len(dataset[s])):
            if dataset[s][f] < mini:
                mini = dataset[s][f]
    
    return mini

In [ ]:
# normalizes the whole dataset
def normalizeUE(dataset):
    maxi = maximum(dataset)
    mini = minimum(dataset)
    N_dataset = np.copy(dataset)
    for s in range(dataset.shape[0]):
        for f in range(len(dataset[s])):
            N_dataset[s][f] = (dataset[s][f] - mini)/(maxi - mini)
    return N_dataset

In [ ]:
# normalizes the whole dataset without taking into account some peaks from the start and end.
def normalizeSE(dataset, fromStart, fromEnd, pad):
    count = MapTool.getPeaks(dataset[0], pad).shape[0]
    start = fromStart
    end = count-1-fromEnd
    
    tempDataset = getSeqSignals(dataset, start, end, pad)
    
    N_dataset = normalizeByOther(dataset, tempDataset)

    return N_dataset

In [ ]:
# normalizes the whole dataset without taking into account some peaks from the start and end.
def normalizeSec(dataset, start, end, pad, convert=""):
    if convert == "":
        convertType = 0
    elif convert == "average":
        convertType = 1
    elif convert == "median":
        convertType = 2
    
    tempDataset = getSeqSignals(dataset, start, end, pad)
    
    N_dataset = normalizeByOther(dataset, tempDataset, pad, convertType)

    return N_dataset

In [ ]:
'''
Normalizes the dataset by taking into account only a section of the dataset.
Then the max and min values are determined by
'''
def normalizeByOther(convertDataset, limitDataset, pad, convertType=0):
    
    if convertType == 0:
        maxi = maximum(limitDataset)
        mini = minimum(limitDataset)
    
    elif convertType == 1:
        Peaks = MapTool.getPeaksDataset(limitDataset, pad)
    
        avg = []

        for p in range(len(Peaks[0])):
            avg.append(np.average(Peaks[:,p]))

        avg = np.array(avg)
    
        maxi = np.max(avg)
        mini = 0
        
    elif convertType == 2:
        Peaks = MapTool.getPeaksDataset(limitDataset, pad)

        median = []

        for p in range(len(Peaks[0])):
            median.append(np.median(Peaks[:,p]))

        median = np.array(median)
    
        maxi = np.max(median)
        mini = 0
    
    print(maxi)
    print(mini)
    
    N_dataset = np.copy(convertDataset)
    for s in range(convertDataset.shape[0]):
        for f in range(len(convertDataset[s])):
            N_dataset[s][f] = (convertDataset[s][f] - mini)/(maxi - mini)
            
    return N_dataset

In [ ]:
'''
def rollingEuclidean_Peaks(FullSignal, SectionSignals, lowestMin, pad, amountPeaks, topNumCorr):
        
    rollingEuclideanList = []
    
    tempSect = MapTool.getPeaks(SectionSignals[0], pad)

    peakLengthSec = MapTool.getPeaks(SectionSignals[0], pad).shape[0] # 1 for the start and end points
    
    peaks = MapTool.getPeaks(FullSignal, pad).shape[0] - MapTool.getPeaks(SectionSignals[0],pad).shape[0] +1
    
    for i in range(peaks):
        j = MapTool.getPeaksLoc(FullSignal, pad, i)
        k = MapTool.getPeaksLoc(FullSignal, pad, i+amountPeaks-1)
        
        part = FullSignal[j:k]
        
        part_Peaks = MapTool.getPeaks(part, pad)

        corrs = []

        for s in range(SectionSignals.shape[0]):           
            sect = SectionSignals[s]
            SectSignal_Peaks = MapTool.getPeaks(sect, pad)

            distSect = distance.euclidean(part_Peaks, SectSignal_Peaks)
            distSect = distSect * -1

            # collect the correlation score.
            corrs.append(distSect)

        # go through various top number of corr to collect the sum and adverage.
        corrs = np.array(corrs)
        topCorrsInd = np.argpartition(corrs, -topNumCorr)[-topNumCorr:]
        topCorrs = corrs[topCorrsInd]
        topCorrsInd = topCorrs.argsort()[:][::-1]
        topCorrs = topCorrs[topCorrsInd]
        
        topCorr = np.mean(topCorrs)
        
        rollingEuclideanList.append(topCorr)
        
    rollingEuclideanList = np.array(rollingEuclideanList, dtype=object)
    
    Min = lowestMin
    Max = 0
    
    rollingEuclideanBestCorrelation = NormalizeData(rollingEuclideanList, Min, Max)
    
    return rollingEuclideanBestCorrelation  
'''

In [ ]:
'''
Method for Normalized Euclidean Distance. This is from the following formula.
normalized Squared Euclidean Distance (NSED): 0.5 * (var(X-Y))/(var(X)+ var(Y))  where var(X) = abs((X-mean(X))^2)
normalized Euclidean Distance: NSED ** 0.5

X: numpy array of values.
Y: numpy array of values.
'''
def N_EuclideanDistance(X, Y):
    nsed = 0.5*((np.var(X-Y))/(np.var(X)+np.var(Y)))
    ned = nsed ** 0.5
    
    return ned

In [ ]:
'''
Method for Euclidean Distance. Makes use of SciPy formula

X: numpy array of values.
Y: numpy array of values.
'''
def EuclideanDistance(X, Y):
    ed = distance.euclidean(X, Y)
    
    return ed

In [ ]:
'''
Method for cosine similarity. Makes use of SciPy formula.

X: numpy array of values.
Y: numpy array of values.
'''
def cosineSimilarity(X, Y):
    cs = distance.cosine(X, Y)
    
    return cs

In [ ]:
'''
This method is used to perform Euclidean Distance on certain peak. 

TS_Signal: (Time Series Signal) the EM signal of an execution path.
Q_Signals: (Query Signal) the EM signals of a set of instructions.
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
peakLoc: the location of the peak to do Euclidean Distance on.
'''
def ED_Peak(TS_Signal, Q_Signals, MM_Scaler, Pad, peak, toNormalize):
    # Create a list of the Euclidean Distances.
    ED_List = []
    
    # Obtain the peak of the TS_Signal to do similarity.
    peakLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peak)
    
    TS_Peak = TS_Signal[peakLoc]
    
    # Go through each Query Signal
    for s in range(Q_Signals.shape[0]):    
        # Select a Query Signal and get only the peak.
        Q_Signal = Q_Signals[s]
        Q_Signal_PeakLoc = MapTool.getPeaksLoc(Q_Signal, Pad, peak)
        Q_Peak = Q_Signal[Q_Signal_PeakLoc]
        
        if toNormalize == 1:
            # perform Min Max scaling on part Time Series Signal and Query Signal
            MM_TS_Peaks = MM_Scaler.transform(np.reshape(TS_Signal, (-1,1)))
            MM_Q_Signal_Peaks = MM_Scaler.transform(np.reshape(Q_Signal, (-1,1)))
            
            TS_Peak = MM_TS_Peaks[peakLoc]
            Q_Peak = MM_Q_Signal_Peaks[Q_Signal_PeakLoc]
            
        # Perform Euclidean Distance on the peaks.
        dist = EuclideanDistance(np.array([TS_Peak]), np.array([Q_Peak]))
        
        # collect the Euclidean Distance score.
        ED_List.append(dist)
    
    return ED_List

In [ ]:
'''
This method is used to perform Euclidean Distance on certain peak. 

Assumes the peaks are given

TS_Peak: (Time Series Signal) the EM signal of an execution path.
Q_Peaks: (Query Signal) the EM signals of a set of instructions.
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
peakLoc: the location of the peak to do Euclidean Distance on.
'''
def ED_Peak_OnlyPeaks(TS_Peak, Q_Peaks, MM_Scaler, Pad, peak, toNormalize):
    # Create a list of the Euclidean Distances.
    ED_List = []
    
    # Obtain the peak of the TS_Peak to do similarity.    
    TS_Peak_value = TS_Peak[peak]
    
    # Go through each Query Signal
    for s in range(Q_Peaks.shape[0]):    
        # Select a Query Signal and get only the peak.
        Q_Peak = Q_Peaks[s]
        Q_Peak_value = Q_Peak[peak]
            
        # Perform Euclidean Distance on the peaks.
        dist = EuclideanDistance(np.array([TS_Peak_value]), np.array([Q_Peak_value]))
        
        # collect the Euclidean Distance score.
        ED_List.append(dist)
    
    return ED_List

In [ ]:
'''
This method is used to perform sliding window similarities. 
The similarities available is 
0) flipped normalized Euclidean Distance: uses euclidean distance then negatives the values. Then Min Max.
1) Normalized Euclidean Distance.
2) Cosine SImilarity
3) Min-Max scale preprocessing Euclidean Distance.

TS_Signal: (Time Series Signal) the EM signal of an execution path.
Q_Signals: (Query Signal) the EM signals of a set of instructions.
MM_Scaler: a MinMax scaler already fitted to the peaks of the time series siganls and Query Signals
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
amountPeaks: The number of peaks that the Query Signal has. 
typeSimM: The type of similarity measurment done
    0) flipped normalized Euclidean Distance: uses euclidean distance then negatives the values. Then Min Max.
    1) Normalized Euclidean Distance.
    2) Cosine SImilarity
    3) Min-Max scale preprocessing Euclidean Distance.
topNumSim: The number of similarities averaged together. Goes by the top amounts. 
'''
def slidingWIndow_Peaks(TS_Signal, QE_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, topNumSim, nonPeakEnds = False,
                       PeakOnlyData = False):
    
    # Create a list of the similarities.
    slidingWindowList = []
    
    # number of peaks to check though on the Time Series Signal. Note that it from start to the end - length of the Query Signal.
    if PeakOnlyData:
        peaks = TS_Signal.shape[0] - QE_Signals[0].shape[0]+1
    else:
        peaks = MapTool.getPeaks(TS_Signal, Pad).shape[0] - MapTool.getPeaks(QE_Signals[0],Pad).shape[0] +1
    
    # Go through each peak of the Time Series Signal.
    for i in range(peaks):       
        if nonPeakEnds:
            # Obtain the section of the TS_Signal to do similarity.
            if PeakOnlyData:
                k = i+amountPeaks-2
            else:
                k = MapTool.getPeaksLoc(TS_Signal, Pad, i+amountPeaks-2)+1
        
            part = TS_Signal[:k]
        
            Q_Signals = []
            for signal in QE_Signals:
                if PeakOnlyData:
                    k = i+amountPeaks-2
                else:
                    k = MapTool.getPeaksLoc(signal, Pad, i+amountPeaks-2)+1
                
                Q_Signals.append(signal[:k])
            
            Q_Signals = np.array(Q_Signals, dtype=object)
            
        else:
            # Obtain the section of the TS_Signal to do similarity.
            if PeakOnlyData:
                j = i
                k = i+amountPeaks
            else:
                j = MapTool.getPeaksLoc(TS_Signal, Pad, i)
                k = MapTool.getPeaksLoc(TS_Signal, Pad, i+amountPeaks)+1

            part = TS_Signal[j:k]
        
            Q_Signals = np.copy(QE_Signals)
        
        # Convert to peaks only for the Time Series signal
        if PeakOnlyData:
            part_Peaks = part
        else:
            part_Peaks = MapTool.getPeaks(part, Pad)
    
        # Create a list for the similarity between Time Series signal and all the Query Signals.
        sims = []

        # Go through each Query Signal
        for s in range(Q_Signals.shape[0]):    
            # Select a Query Signal and get only the peaks.
            Q_Signal = Q_Signals[s]
            if PeakOnlyData:
                Q_Signal_Peaks = Q_Signal
            else:
                Q_Signal_Peaks = MapTool.getPeaks(Q_Signal, Pad)

            # Perform one of the sliding window similarity options.
            ## 0) performs flipped normalized Euclidean Distance
            if typeSimM == 0:
                dist = distance.euclidean(part_Peaks, Q_Signal_Peaks)
                dist = dist * -1
            
            ## 1) performs Normalized Euclidean Distance
            elif typeSimM == 1:
                dist = N_EuclideanDistance(part_Peaks, Q_Signal_Peaks)
            
            ## 2) performs Cosine Similarity
            elif typeSimM == 2:
                dist = cosineSimilarity(part_Peaks, Q_Signal_Peaks)
            
            ## 3) performs Min-Max scale preprocessing Euclidean Distance
            elif typeSimM == 3:
                # perform Min Max scaling on part Time Series Signal and Query Signal
                MM_part_Peaks = MM_Scaler.transform(np.reshape(part_Peaks, (-1,1)))
                MM_Q_Signal_Peaks = MM_Scaler.transform(np.reshape(Q_Signal_Peaks, (-1,1)))
                
                MM_part_Peaks = MM_part_Peaks.flatten()
                MM_Q_Signal_Peaks = MM_Q_Signal_Peaks.flatten()
                
                dist = EuclideanDistance(MM_part_Peaks, MM_Q_Signal_Peaks)
            
            ## 4) basic Euclidean Distance
            elif typeSimM == 4:
                dist = EuclideanDistance(part_Peaks, Q_Signal_Peaks)
                

            # collect the similarity score.
            sims.append(dist)

        # go through various top number of similarities and obtain the mean.
        ## this is to reduce the effect of outlier query signals.
        ## This is for getting the top distances.
        #sims = np.array(sims)
        #print(sims)
        #topSimsInd = np.argpartition(sims, -topNumSim)[-topNumSim:]
        #topSims = sims[topSimsInd]
        #topSimsInd = topSims.argsort()[:][::-1]
        #topSims = topSims[topSimsInd]
        #print(topSims)
        
        #topSim = np.mean(topSims)
        ## this is for getting the lowest distances.
        sims = np.array(sims)
        topSimsInd = np.argpartition(sims, topNumSim)[:topNumSim]
        topSims = sims[topSimsInd]
        topSimsInd = topSims.argsort()[:][::1]
        topSims = topSims[topSimsInd]
        
        topSim = np.mean(topSims)
        
        # place inside a list of the similarities
        slidingWindowList.append(topSim)
        
    slidingWindowList = np.array(slidingWindowList, dtype=object)
    
    # last part of flipped normalized Euclidean Distance.
    ## this normalizes the similarities to a set standard for Min and Max euclidean distance.
    if typeSimM == 0:
        Min = -200
        Max = 0
    
        slidingWindowList = NormalizeData(slidingWindowList, Min, Max)
    
    return slidingWindowList

In [ ]:
def plotSimilarity(similarity, titleName, toFileSave, location):
    ### Plot the correlation
    plt.plot(similarity)
    plt.title(titleName)
    plt.xlabel("Starting Peak in Time Sereies Signal")
    plt.ylabel("Similarity Measurement")
    if toFileSave == 0:
        plt.show()
    elif toFileSave == 1:
        try:
            plt.savefig(location)
        except FileNotFoundError as error:
            logging.error(error)
    elif toFileSave == 2:
        try:
            plt.savefig(location)
        except FileNotFoundError as error:
            logging.error(error)
        plt.show()

In [ ]:
def plotMostLikelyArea(TS_Signal, Q_Signals, sims, TS_Signal_name, Q_Signal_name, titleName, pad, amountPeaks, 
                       toFileSave, location):
    ### Plot the most likely area in the signal sample

    peakNum = np.argmin(sims)
    start = MapTool.getPeaksLoc(TS_Signal, pad, peakNum)
    sim = min(sims)
    end = MapTool.getPeaksLoc(TS_Signal, pad, peakNum+amountPeaks-1)
    QInSample = TS_Signal[start:end]

    time_Signal = np.arange(TS_Signal.shape[0])
    time_Seq = np.arange(start,end)

    plt.plot(time_Signal, TS_Signal, label = "Time-Series " + TS_Signal_name)
    plt.plot(time_Seq, QInSample, label = "Query " +  Q_Signal_name)
    plt.title(titleName)
    plt.xlabel("Sample Index")
    plt.ylabel("Amplitude")
    plt.legend()
    if toFileSave == 0:
        plt.show()
    elif toFileSave == 1:
        try:
            plt.savefig(location)
        except FileNotFoundError as error:
            logging.error(error)
    elif toFileSave == 2:
        try:
            plt.savefig(location)
        except FileNotFoundError as error:
            logging.error(error)
        plt.show()

In [ ]:
def getSeqSignals(Signals, peaksStart, peaksEnd, pad):
    seqList = []

    for signal in Signals:    
        Seq = signal[MapTool.getPeaksLoc(signal, pad, peaksStart):MapTool.getPeaksLoc(signal, pad, peaksEnd)]
        seqList.append(Seq)
    
    seqList = np.array(seqList, dtype="object")
    return seqList

In [ ]:
'''
def getEvaluations(corrs, ground_truths, numberThresholds=1000):
    results = []

    for threshold in np.arange(0, 1 + 1 / numberThresholds, 1 / numberThresholds):
        predictions = []
        for corr in corrs:
            for index in range(corr.shape[0]):
                if corr[index] > threshold:
                    predictions.append(1)
                else:
                    predictions.append(0)
        
        predictions = np.array(predictions)
        
        ground_truth = ground_truths.flatten()

        metrics = ConfidenceEstimation.calculateBasicMetrics(ground_truth, predictions)

        results.append(metrics)

    return results
'''

In [ ]:
def getEvaluations(sims, ground_truths, numberThresholds=1000):
    results = []

    for threshold in np.arange(0, 1 + 1 / numberThresholds, 1 / numberThresholds):
        predictions = []
        for sim in sims:
            for index in range(sim.shape[0]):
                if sim[index] > threshold:
                    predictions.append(0)
                else:
                    predictions.append(1)
        
        predictions = np.array(predictions)
        
        ground_truth = ground_truths.flatten()

        metrics = ConfidenceEstimation.calculateBasicMetrics(ground_truth, predictions)

        results.append(metrics)

    return results

In [ ]:
def getEvaluationsOne(corr, ground_truth, numberThresholds=1000):
    results = []

    for threshold in np.arange(0, 1 + 1 / numberThresholds, 1 / numberThresholds):
        predictions = []
        for index in range(corr.shape[0]):
            if corr[index] > threshold:
                predictions.append(1)
            else:
                predictions.append(0)
        
        predictions = np.array(predictions)

        metrics = ConfidenceEstimation.calculateBasicMetrics(ground_truth, predictions)

        results.append(metrics)

    return results

In [ ]:
# Methods for writing to a file. Two exists. One that will only write to a file and one that will write to file and output
# to jupyter Notebook.

def printOrWriteInfo(location, text, toFileSave, firstLine = False):
    if toFileSave ==0:
        print(text)
    elif toFileSave == 1:
        if firstLine:
            sourceFile = open(location, 'w')
        else:
            sourceFile = open(location, 'a')
        sourceFile.write(text)
        sourceFile.close()
    elif toFileSave == 2:
        if firstLine:
            sourceFile = open(location, 'w')
        else:
            sourceFile = open(location, 'a')
        sourceFile.write(text)
        print(text)
        sourceFile.close()

In [ ]:
'''
Performs the experiment of comparing the instructions main peaks to determin the number of peaks influenced by prior instruction/s.

TS_Signals_G: (Time Series Signals) the EM signals of an execution path.
Q_Signals_G: (Query Signals) the EM signals of an set instruction/s.
TS_Signal_name: the name of the time series signal to be displayed in graphs
Q_Signal_name: the name of the query signal to be displayed in graphs
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
dictionary: contains information on the instructions. (peakLoc, name)
'''

def instComparisonExp(TS_Signals_G, Q_Signals_G, TS_Signal_name, Q_Signal_name, Pad, dictionary,
                     y_limit_low=0, y_limit_high=0, toNormalize = 0):
    
    TS_Signals = TS_Signals_G
    Q_Signals = Q_Signals_G
    
    # Create the Min-Max Scalar
    MM_Scaler = MinMaxScaler()
    
    # normalize the two datasets
    if toNormalize == 1:       
        # gather all the peaks of the Time Series Signals and Query Signals
        TS_Signals_Peaks =  MapTool.getPeaksDataset(TS_Signals, Pad)
        Q_Signals_Peaks =  MapTool.getPeaksDataset(Q_Signals, Pad)
        
        # Find the max and min of the two.
        TS_Max = np.max(TS_Signals_Peaks)
        Q_Max = np.max(Q_Signals_Peaks)
        TS_Min = np.min(TS_Signals_Peaks)
        Q_Min = np.mmin(Q_Signals_Peaks)
        
        # place into array
        info = np.array([TS_Max, Q_Max, TS_Min, Q_Min])
        info = np.reshape(info, (-1,1)) 
        
        # Create the Min-Max Scalar based on the min max of the info collected.
        MM_Scaler.fit(info)
    
    print("Beginning Tests...")
    ED_List_All =[]
    
    # Gather the Euclidean Distance of each instruction comparison
    ## go through each instruction
    for i in range(dictionary.shape[0]):
        peaks = dictionary[i][0]
        name =  dictionary[i][1]
        
        print("Working on Instruction: "+ name)
        start = timeit.default_timer()
        
        ED_Inst_List = []
        
        # Go through each Time-Series Sample
        for t in range(TS_Signals.shape[0]):

            TS_Signal = TS_Signals[t]

            # Obtain the Euclidean DIstances across the Time-Series Sample given the Query signals for the peaks
            ED_TS_Sample_List = ED_Peak(TS_Signal, Q_Signals, MM_Scaler, Pad, peaks, toNormalize)
            ED_TS_Sample_List = np.array(ED_TS_Sample_List)
            
            for p in range(ED_TS_Sample_List.shape[0]):
                ED_Inst_List.append(ED_TS_Sample_List[p])
        
        ED_Inst_List = np.array(ED_Inst_List)
        print(ED_Inst_List.shape)
        
        rowInfo = np.array([peaks, name, ED_Inst_List], dtype=object)
        ED_List_All.append(rowInfo)
        
        stop = timeit.default_timer()

        print('Time: ', stop - start)  
            
    ED_List_All = np.array(ED_List_All)
    
    # Plot the max Euclidean Distance distance of all the instructions
    plotDistances = []
    plotXNames = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.max(ED_List_All[n][2]))
        plotXNames.append(ED_List_All[n][1])
    
    plotDistances = np.array(plotDistances)
    plotXNames = np.array(plotXNames)
    
    titleName = "Maximum Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    
    # Plot the average Euclidean Distance of all the instructions.
    plotDistances = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.mean(ED_List_All[n][2]))
    
    plotDistances = np.array(plotDistances)
    
    titleName = "Average Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    # Plot the min Euclidean Distance distance of all the instructions.
    plotDistances = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.min(ED_List_All[n][2]))
    
    plotDistances = np.array(plotDistances)
    
    titleName = "Minimum Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    # Plot the BoxGraph of the Euclidean Distance of all instructions.
    titleName = "BoxPlot of the EDs between the TS and Q Samples"
    BoxPlot.boxPlotInstPeaks(ED_List_All, "Instructions", "Euclidean Distance", titleName, y_limit_low, y_limit_high)
    

In [ ]:
'''
Performs the experiment of comparing the instructions main peaks to determin the number of peaks influenced by prior instruction/s.

This Method assumes the Peaks have alread been taken

TS_Peaks_G: (Time Series Signals) the EM signals of an execution path.
Q_Peaks_G: (Query Signals) the EM signals of an set instruction/s.
TS_Peak_name: the name of the time series signal to be displayed in graphs
Q_Peak_name: the name of the query signal to be displayed in graphs
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
dictionary: contains information on the instructions. (peakLoc, name)
'''

def instComparisonExp_OnlyPeaks(TS_Peaks_G, Q_Peaks_G, TS_Peak_name, Q_Peak_name, Pad, dictionary,
                     y_limit_low=0, y_limit_high=0, toNormalize = 0):
    
    TS_Peaks = TS_Peaks_G
    Q_Peaks = Q_Peaks_G
    
    # Create the Min-Max Scalar
    MM_Scaler = MinMaxScaler()
    
    
    print("Beginning Tests...")
    ED_List_All =[]
    
    # Gather the Euclidean Distance of each instruction comparison
    ## go through each instruction
    for i in range(dictionary.shape[0]):
        peaks = dictionary[i][0]
        name =  dictionary[i][1]
        
        print("Working on Instruction: "+ name)
        start = timeit.default_timer()
        
        ED_Inst_List = []
        
        # Go through each Time-Series Sample
        for t in range(TS_Peaks.shape[0]):

            TS_Peak = TS_Peaks[t]

            # Obtain the Euclidean DIstances across the Time-Series Sample given the Query signals for the peaks
            ED_TS_Sample_List = ED_Peak_OnlyPeaks(TS_Peak, Q_Peaks, MM_Scaler, Pad, peaks, toNormalize)
            ED_TS_Sample_List = np.array(ED_TS_Sample_List)
            
            for p in range(ED_TS_Sample_List.shape[0]):
                ED_Inst_List.append(ED_TS_Sample_List[p])
        
        ED_Inst_List = np.array(ED_Inst_List)
        print(ED_Inst_List.shape)
        
        rowInfo = np.array([peaks, name, ED_Inst_List], dtype=object)
        ED_List_All.append(rowInfo)
        
        stop = timeit.default_timer()

        print('Time: ', stop - start)  
            
    ED_List_All = np.array(ED_List_All)
    
    # Plot the max Euclidean Distance distance of all the instructions
    plotDistances = []
    plotXNames = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.max(ED_List_All[n][2]))
        plotXNames.append(ED_List_All[n][1])
    
    plotDistances = np.array(plotDistances)
    plotXNames = np.array(plotXNames)
    
    titleName = "Maximum Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    
    # Plot the average Euclidean Distance of all the instructions.
    plotDistances = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.mean(ED_List_All[n][2]))
    
    plotDistances = np.array(plotDistances)
    
    titleName = "Average Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    # Plot the min Euclidean Distance distance of all the instructions.
    plotDistances = []
    
    for n in range(ED_List_All.shape[0]):
        plotDistances.append(np.min(ED_List_All[n][2]))
    
    plotDistances = np.array(plotDistances)
    
    titleName = "Minimum Distances between the TS and Q Samples"
    
    plt.plot(plotDistances)
    plt.title(titleName)
    
    plt.xlabel("Instructions")
    plt.ylabel("Euclidean Distance")
    
    x = np.arange(plotDistances.shape[0])
    
    plt.xticks(x, plotXNames)
    # Pad margins so that markers don't get clipped by the axes
    plt.margins(0.2)
    # Tweak spacing to prevent clipping of tick-labels
    plt.subplots_adjust(bottom=0.15)
    plt.show()
    
    # Plot the BoxGraph of the Euclidean Distance of all instructions.
    titleName = "BoxPlot of the EDs between the TS and Q Samples"
    BoxPlot.boxPlotInstPeaks(ED_List_All, "Instructions", "Euclidean Distance", titleName, y_limit_low, y_limit_high)

In [ ]:
'''
Performs experiments with sliding window similarities between a given Time Series Signals and Query Signals.

TS_Signals: (Time Series Signals) the EM signals of an execution path.
Q_Signals: (Query Signals) the EM signals of an set instruction/s.
TS_Signal_name: the name of the time series signal to be displayed in graphs
Q_Signal_name: the name of the query signal to be displayed in graphs
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
typeSimM: The type of similarity measurment done.
    0) flipped normalized Euclidean Distance: uses euclidean distance then negatives the values. Then Min Max.
    1) Normalized Euclidean Distance.
    2) Cosine SImilarity
    3) Min-Max scale preprocessing Euclidean Distance.
peaksToLookAt: the peaks that contains the Query signal in the given Time Series Signals.
    So far only works with ONE.
amtGraphs: the amount of sliding Window similarity Graphs to show.
topNumSim: the number of sliding Window similarities to mean for one similarity. To reduce the effects of outlier Query signals.
    Note given as array to test multiple values
'''
def slidingWindowExp(TS_Signals, Q_Signals, TS_Signal_name, Q_Signal_name, Pad, peaksToLookAt, typeSimM,
                     amtGraphs = 20, topNumSim=np.array([10]), toFileSave = 0, GeneralPath= ""):
    
    if toFileSave < 0 or toFileSave > 2:
        print("Error: non valid toFileSave value.")
        
    if toFileSave == 1 or toFileSave == 2:
        if GeneralPath == "":
            print("Error: no path specified.")
    
    ResultsFile = GeneralPath + "/results.txt"
    
    ## Create path if does not exist.
    # Check whether the specified path exists or not
    isExist = os.path.exists(GeneralPath)

    if not isExist:
      # Create a new directory because it does not exist 
      os.makedirs(GeneralPath)
    
    # Get the number of peaks in the Query Signals
    amountPeaks = MapTool.getPeaks(Q_Signals[0], Pad).shape[0]
    
    # Create the Min-Max Scalar
    MM_Scaler = MinMaxScaler()
    
    # String name of the similarity measurement.
    SM_name = ""
    if typeSimM == 0:
        SM_name = "flipped normalized Euclidean Distance"
        SM_nameShort = "FNED"
    elif typeSimM == 1:
        SM_name = "Normalized Euclidean Distance"
        SM_nameShort = "NED"
    elif typeSimM == 2:
        SM_name = "Cosine Similarity"
        SM_nameShort = "CS"
    elif typeSimM == 3:
        SM_name = "MinMax Euclidean Distance"
        SM_nameShort = "MED"
        
    ResultsFile = GeneralPath + SM_nameShort + "_TS_" + TS_Signal_name + "_Q_" + Q_Signal_name + " results.txt"
    
    printOrWriteInfo(ResultsFile, "Beginning Tests...", toFileSave, True)
    
    # check for errors:
    checkCount = []
    for Q_Signal in Q_Signals:
        checkCount.append(MapTool.getPeaks(Q_Signal, Pad).shape[0])
    
    checkCount = np.array(checkCount)
    checkCount = np.unique(checkCount)
    
    if checkCount.shape[0] > 1:
        printOrWriteInfo(ResultsFile, "Error in gathering Q_Signals as unique peaks gave different values", toFileSave)
        printOrWriteInfo(ResultsFile, checkCount, toFileSave)
    
    
    # Create a Min Max Scaler for later use with Min-Max scale preprocessing Euclidean Distance
    if typeSimM == 3:
        # gather all the peaks of the Time Series Signals and Query Signals
        TS_Signals_Peaks =  MapTool.getPeaksDataset(TS_Signals, Pad)
        Q_Signals_Peaks =  MapTool.getPeaksDataset(Q_Signals, Pad)
        
        # Find the max and min of the two.
        TS_Max = np.max(TS_Signals_Peaks)
        Q_Max = np.max(Q_Signals_Peaks)
        TS_Min = np.min(TS_Signals_Peaks)
        Q_Min = np.min(Q_Signals_Peaks)
        
        # place into array
        info = np.array([TS_Max, Q_Max, TS_Min, Q_Min])
        info = np.reshape(info, (-1,1)) 
        
        # Create the Min-Max Scalar based on the min max of the info collected.
        MM_Scaler.fit(info)
    
    # Set the number of the top number of sliding Window similarities.
    ## this will go through each to test each number given
    for number in topNumSim:
        start = timeit.default_timer()
    
        count = 0
        correct =  0
        SW_Sims = []
        
        
        printOrWriteInfo(ResultsFile, " ", toFileSave)
        printOrWriteInfo(ResultsFile, "Taking the top number of sliding Window similarities: " + str(number), toFileSave)
        
        # Get a Time Series Signal for similarity measurement
        for i in range(TS_Signals.shape[0]):

            TS_Signal = TS_Signals[i]

            # Find out if to graph the sliding window similarity measurement for the time series signal
            ## This case graph
            if i < amtGraphs:
                # Title of section
                if toFileSave == 0 or toFileSave == 2:
                    print(TS_Signal_name + ": Signal Sample " + str(i))

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number)
                SW_Sims.append(sims)

                ### Plot the correlation
                simFile = "Sim_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotSimilarity(sims, SM_name + " between TS from " + TS_Signal_name + " Sample: " + str(i) + 
                               " and Q from " + Q_Signal_name,
                               toFileSave, GeneralPath + "Graphs/" + simFile)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)
                mostLikelyFile = "MLArea_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotMostLikelyArea(TS_Signal, Q_Signals[0], sims, TS_Signal_name, Q_Signal_name, 
                                   "Best " + SM_name + " for " + Q_Signal_name + ": " + str(sim_best), Pad, amountPeaks,
                                  toFileSave, GeneralPath + "Graphs/" + mostLikelyFile)

                if toFileSave == 0 or toFileSave == 2:
                    print("Found best peak: ", np.argmin(sims))

                    # Spacing for visability
                    print("============================================================= ")
                    print(" ")
                    print("============================================================= ")

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            ## This case do not graph.
            else:
                if i%10==0:
                    if toFileSave == 0 or toFileSave == 2:
                        print("Working on Sample:", i)

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number)
                SW_Sims.append(sims)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            count = count + 1

        SW_Sims =np.array(SW_Sims)

        peakToLookAt
        
        # Note so far only works with ONE actual peak in Time Series. 
        ground_truth = np.zeros((SW_Sims.shape[0], SW_Sims.shape[1]))
        
        for actualPeak in peaksToLookAt:
            ground_truth[np.arange(SW_Sims.shape[0]), actualPeak] = 1
        

        list_of_metrics = getEvaluations(SW_Sims, ground_truth, 1000)

        accumulative_metrics = ConfidenceEstimation.calculateMultipleRoundMetrics(list_of_metrics)

        auc_Score = accumulative_metrics["auc"]

        # Gather data
        fprs = [item["fall_out"] for item in list_of_metrics]
        tprs = [item["sensitivity"] for item in list_of_metrics]

        fprs = np.array(fprs)
        tprs = np.array(tprs)

        accsSec = [item["acc"] for item in list_of_metrics]

        printOrWriteInfo(GeneralPath, "Results for Sliding Window Similiarity between the Time Series Signals " +
                         TS_Signal_name + " and the Query Signals " + Q_Signal_name, toFileSave)
        rocFile = "ROC_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Q_" + Q_Signal_name
        rocTitle = "ROC for" + SM_name + "Between " + TS_Signal_name + " and " + Q_Signal_name
        aucScoreSave = int(auc_Score * 1000)

        if toFileSave == 0:
            ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score)
        elif toFileSave == 1:
            ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score, GeneralPath + "Graphs/" +
                       rocFile )
        elif toFileSave == 2:
            ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score)
            ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score, GeneralPath + "Graphs/" +
                       rocFile )
            
        printOrWriteInfo(ResultsFile, "ROC over all samples: " + str(auc_Score), toFileSave)
        printOrWriteInfo(ResultsFile, "Accuracy over all samples correctly labeled: " + str(correct/count), toFileSave)
        printOrWriteInfo(ResultsFile, "Accuracy over all labels: " + str(max(accsSec)), toFileSave)

        stop = timeit.default_timer()

        print('Time: ', stop - start)  

In [ ]:
'''
Performs experiments with sliding window similarities between a given Time Series Signals segment and Query Signals.

Note: this is only to give general idea of how similar they are. Gives just the average similarity.
Note: the Time-Series signal segment and Queries must have the same number of peaks.

TS_Signals: (Time Series Signals) the EM signals of an execution path.
Q_Signals: (Query Signals) the EM signals of an set instruction/s.
TS_Signal_name: the name of the time series signal to be displayed in graphs
Q_Signal_name: the name of the query signal to be displayed in graphs
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
typeSimM: The type of similarity measurment done.
    0) flipped normalized Euclidean Distance: uses euclidean distance then negatives the values. Then Min Max.
    1) Normalized Euclidean Distance.
    2) Cosine SImilarity
    3) Min-Max scale preprocessing Euclidean Distance.
peaksToLookAt: the peaks that contains the Query signal in the given Time Series Signals.
    So far only works with ONE.
amtGraphs: the amount of sliding Window similarity Graphs to show.
topNumSim: the number of sliding Window similarities to mean for one similarity. To reduce the effects of outlier Query signals.
    Note given as array to test multiple values
'''
def averageSimilarityExp(TS_Signals, Q_Signals, TS_Signal_name, Q_Signal_name, Pad, peaksToLookAt, typeSimM,
                     amtGraphs = 20, topNumSim=np.array([10]), toFileSave = 0, GeneralPath= ""):
    
    if toFileSave < 0 or toFileSave > 2:
        print("Error: non valid toFileSave value.")
        
    if toFileSave == 1 or toFileSave == 2:
        if GeneralPath == "":
            print("Error: no path specified.")
    
    ResultsFile = GeneralPath + "/results.txt"
    
    ## Create path if does not exist.
    # Check whether the specified path exists or not
    isExist = os.path.exists(GeneralPath)

    if not isExist:
      # Create a new directory because it does not exist 
      os.makedirs(GeneralPath)
    
    # Get the number of peaks in the Query Signals
    amountPeaks = MapTool.getPeaks(Q_Signals[0], Pad).shape[0]
    amountPeaksTS = MapTool.getPeaks(TS_Signals[0], Pad).shape[0]
    amountPeaksQ = MapTool.getPeaks(Q_Signals[0], Pad).shape[0]
    
    if amountPeaksTS != amountPeaksQ:
        print("Error: differing numnber of peaks.")
    
    # Create the Min-Max Scalar
    MM_Scaler = MinMaxScaler()
    
    # String name of the similarity measurement.
    SM_name = ""
    if typeSimM == 0:
        SM_name = "flipped normalized Euclidean Distance"
        SM_nameShort = "FNED"
    elif typeSimM == 1:
        SM_name = "Normalized Euclidean Distance"
        SM_nameShort = "NED"
    elif typeSimM == 2:
        SM_name = "Cosine Similarity"
        SM_nameShort = "CS"
    elif typeSimM == 3:
        SM_name = "MinMax Euclidean Distance"
        SM_nameShort = "MED"
        
    ResultsFile = GeneralPath + SM_nameShort + "_TS_" + TS_Signal_name + "_Q_" + Q_Signal_name + " results.txt"
    
    printOrWriteInfo(ResultsFile, "Beginning Tests...", toFileSave, True)
    
    # check for errors:
    checkCount = []
    for Q_Signal in Q_Signals:
        checkCount.append(MapTool.getPeaks(Q_Signal, Pad).shape[0])
    
    checkCount = np.array(checkCount)
    checkCount = np.unique(checkCount)
    
    if checkCount.shape[0] > 1:
        printOrWriteInfo(ResultsFile, "Error in gathering Q_Signals as unique peaks gave different values", toFileSave)
        printOrWriteInfo(ResultsFile, checkCount, toFileSave)
    
    
    # Create a Min Max Scaler for later use with Min-Max scale preprocessing Euclidean Distance
    if typeSimM == 3:
        # gather all the peaks of the Time Series Signals and Query Signals
        if nonPeakEnds:
            # Obtain the section of the TS_Signal to do similarity.
            k = i+amountPeaks-2
        else:
            k = i+amountPeaks
            
        TS_Signals_Peaks = MapTool.getPeaksDataset(TS_Signals, Pad)
        TS_Signals_Peaks = TS_Signals_Peaks[:][:k]
            
        Q_Signals_Peaks =  MapTool.getPeaksDataset(Q_Signals, Pad)
        Q_Signals_Peaks = Q_Signals_Peaks[:][:k]
        
        print(TS_Signals_Peaks[0])
        plt.plot(TS_Signals_Peaks[0])
        plt.show()
        
        print(Q_Signals_Peaks[0])
        plt.plot(Q_Signals_Peaks[0])
        plt.show()
        
        # Find the max and min of the two.
        TS_Max = np.max(TS_Signals_Peaks)
        Q_Max = np.max(Q_Signals_Peaks)
        TS_Min = np.min(TS_Signals_Peaks)
        Q_Min = np.min(Q_Signals_Peaks)
        
        # place into array
        info = np.array([TS_Max, Q_Max, TS_Min, Q_Min])
        info = np.reshape(info, (-1,1)) 
        
        # Create the Min-Max Scalar based on the min max of the info collected.
        MM_Scaler.fit(info)
    
    # Set the number of the top number of sliding Window similarities.
    ## this will go through each to test each number given
    for number in topNumSim:
        start = timeit.default_timer()
    
        count = 0
        correct =  0
        SW_Sims = []
        
        
        printOrWriteInfo(ResultsFile, " ", toFileSave)
        printOrWriteInfo(ResultsFile, "Taking the top number of sliding Window similarities: " + str(number), toFileSave)
        
        # Get a Time Series Signal for similarity measurement
        for i in range(TS_Signals.shape[0]):

            TS_Signal = TS_Signals[i]

            # Find out if to graph the sliding window similarity measurement for the time series signal
            ## This case graph
            if i < amtGraphs:
                # Title of section
                if toFileSave == 0 or toFileSave == 2:
                    print(TS_Signal_name + ": Signal Sample " + str(i))

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number)
                if sims.shape != 1:
                    print("Error: more than one comparison done for this similarity.")
                    print("Similarities calculated: " + str(sims.shape))
                SW_Sims.append(sims)

                ### Plot the correlation
                simFile = "Sim_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotSimilarity(sims, SM_name + " between TS from " + TS_Signal_name + " Sample: " + str(i) + 
                               " and Q from " + Q_Signal_name,
                               toFileSave, GeneralPath + "Graphs/" + simFile)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)
                mostLikelyFile = "MLArea_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotMostLikelyArea(TS_Signal, Q_Signals[0], sims, TS_Signal_name, Q_Signal_name, 
                                   "Best " + SM_name + " for " + Q_Signal_name + ": " + str(sim_best), Pad, amountPeaks,
                                  toFileSave, GeneralPath + "Graphs/" + mostLikelyFile)

                if toFileSave == 0 or toFileSave == 2:
                    print("Found best peak: ", np.argmin(sims))

                    # Spacing for visability
                    print("============================================================= ")
                    print(" ")
                    print("============================================================= ")

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            ## This case do not graph.
            else:
                if i%10==0:
                    if toFileSave == 0 or toFileSave == 2:
                        print("Working on Sample:", i)

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number)
                SW_Sims.append(sims)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            count = count + 1

        SW_Sims =np.array(SW_Sims)
            
        printOrWriteInfo(ResultsFile, "Average Similarity: " + str(np.mean(SW_Sims)), toFileSave)

        stop = timeit.default_timer()

        print('Time: ', stop - start)  

In [2]:
'''
Performs experiments with sliding window similarities between a given Time Series Signals segment and Query Signals.

Note: this is only to give general idea of how similar they are. Gives just the average similarity.
Note: the Time-Series signal segment and Queries must have the same number of peaks.

TS_Signals: (Time Series Signals) the EM signals of an execution path.
Q_Signals: (Query Signals) the EM signals of an set instruction/s.
TS_Signal_name: the name of the time series signal to be displayed in graphs
Q_Signal_name: the name of the query signal to be displayed in graphs
Pad: Used for collecting the peaks. Is the average number of features between high and low peaks.
typeSimM: The type of similarity measurment done.
    0) flipped normalized Euclidean Distance: uses euclidean distance then negatives the values. Then Min Max.
    1) Normalized Euclidean Distance.
    2) Cosine SImilarity
    3) Min-Max scale preprocessing Euclidean Distance.
peaksToLookAt: the peaks that contains the Query signal in the given Time Series Signals.
    So far only works with ONE.
amtGraphs: the amount of sliding Window similarity Graphs to show.
topNumSim: the number of sliding Window similarities to mean for one similarity. To reduce the effects of outlier Query signals.
    Note given as array to test multiple values
'''
def averageSimilarityExpReturn(TS_Signals, Q_Signals, TS_Signal_name, Q_Signal_name, Pad, peaksToLookAt, typeSimM,
                     amtGraphs = 20, topNumSim=np.array([10]), toFileSave = 0, GeneralPath= "", nonPeakEnds = False, Info = True,
                              PeakOnlyData = False):
    
    if toFileSave < 0 or toFileSave > 2:
        print("Error: non valid toFileSave value.")
        
    if toFileSave == 1 or toFileSave == 2:
        if GeneralPath == "":
            print("Error: no path specified.")
    
    ResultsFile = GeneralPath + "/results.txt"
    
    if toFileSave == 1 or toFileSave == 2:
        ## Create path if does not exist.
        # Check whether the specified path exists or not
        isExist = os.path.exists(GeneralPath)

        if not isExist:
          # Create a new directory because it does not exist 
          os.makedirs(GeneralPath)
    
    # Get the number of peaks in the Query Signals
    if PeakOnlyData:
        amountPeaks = Q_Signals[0].shape[0]
        amountPeaksTS = TS_Signals[0].shape[0]
        amountPeaksQ = Q_Signals[0].shape[0]
    else:
        amountPeaks = MapTool.getPeaks(Q_Signals[0], Pad).shape[0]
        amountPeaksTS = MapTool.getPeaks(TS_Signals[0], Pad).shape[0]
        amountPeaksQ = MapTool.getPeaks(Q_Signals[0], Pad).shape[0]

    if amountPeaksTS != amountPeaksQ:
        print("Error: differing numnber of peaks.")
    
    # Create the Min-Max Scalar
    MM_Scaler = MinMaxScaler()
    
    # String name of the similarity measurement.
    SM_name = ""
    if typeSimM == 0:
        SM_name = "flipped normalized Euclidean Distance"
        SM_nameShort = "FNED"
    elif typeSimM == 1:
        SM_name = "Normalized Euclidean Distance"
        SM_nameShort = "NED"
    elif typeSimM == 2:
        SM_name = "Cosine Similarity"
        SM_nameShort = "CS"
    elif typeSimM == 3:
        SM_name = "MinMax Euclidean Distance"
        SM_nameShort = "MED"
    elif typeSimM == 4:
        SM_name = "Euclidean Distance"
        SM_nameShort = "ED"
        
    ResultsFile = GeneralPath + SM_nameShort + "_TS_" + TS_Signal_name + "_Q_" + Q_Signal_name + " results.txt"
    
    if Info:
        printOrWriteInfo(ResultsFile, "Beginning Tests...", toFileSave, True)
    
    # check for errors:
    if PeakOnlyData == False:
        checkCount = []
        for Q_Signal in Q_Signals:
            checkCount.append(MapTool.getPeaks(Q_Signal, Pad).shape[0])

        checkCount = np.array(checkCount)
        checkCount = np.unique(checkCount)

        if checkCount.shape[0] > 1:
            printOrWriteInfo(ResultsFile, "Error in gathering Q_Signals as unique peaks gave different values", toFileSave)
            printOrWriteInfo(ResultsFile, checkCount, toFileSave)
    
    
    # Create a Min Max Scaler for later use with Min-Max scale preprocessing Euclidean Distance
    if typeSimM == 3:
        # gather all the peaks of the Time Series Signals and Query Signals
        if nonPeakEnds:
            # Obtain the section of the TS_Signal to do similarity.
            k = amountPeaks-1
        else:
            k = amountPeaks
        
        if PeakOnlyData:
            TS_Signals_Peaks = TS_Signals
            Q_Signals_Peaks = Q_Signals
        else:   
            TS_Signals_Peaks = MapTool.getPeaksDataset(TS_Signals, Pad)
            Q_Signals_Peaks =  MapTool.getPeaksDataset(Q_Signals, Pad)
        
        TS_Signals_Peaks = TS_Signals_Peaks[:,:k]
        Q_Signals_Peaks = Q_Signals_Peaks[:,:k]
        
        # Find the max and min of the two.
        TS_Max = np.max(TS_Signals_Peaks)
        Q_Max = np.max(Q_Signals_Peaks)
        TS_Min = np.min(TS_Signals_Peaks)
        Q_Min = np.min(Q_Signals_Peaks)
        
        # place into array
        info = np.array([TS_Max, Q_Max, TS_Min, Q_Min])
        info = np.reshape(info, (-1,1)) 
        
        # Create the Min-Max Scalar based on the min max of the info collected.
        MM_Scaler.fit(info)
    
    # Set the number of the top number of sliding Window similarities.
    ## this will go through each to test each number given
    for number in topNumSim:
        start = timeit.default_timer()
    
        count = 0
        correct =  0
        SW_Sims = []
        
        if Info:
            printOrWriteInfo(ResultsFile, " ", toFileSave)
            printOrWriteInfo(ResultsFile, "Taking the top number of sliding Window similarities: " + str(number), toFileSave)
        
        # Get a Time Series Signal for similarity measurement
        for i in range(TS_Signals.shape[0]):

            TS_Signal = TS_Signals[i]

            # Find out if to graph the sliding window similarity measurement for the time series signal
            ## This case graph
            if i < amtGraphs:
                # Title of section
                if toFileSave == 0 or toFileSave == 2:
                    print(TS_Signal_name + ": Signal Sample " + str(i))

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number, nonPeakEnds,
                                          PeakOnlyData)
                if sims.shape != 1:
                    print("Error: more than one comparison done for this similarity.")
                    print("Similarities calculated: " + str(sims.shape))
                SW_Sims.append(sims)

                ### Plot the correlation
                simFile = "Sim_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotSimilarity(sims, SM_name + " between TS from " + TS_Signal_name + " Sample: " + str(i) + 
                               " and Q from " + Q_Signal_name,
                               toFileSave, GeneralPath + "Graphs/" + simFile)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)
                mostLikelyFile = "MLArea_" + SM_nameShort + "_TS_" + TS_Signal_name + "_Sample_" + str(i) + "_Q_" + Q_Signal_name + ".png"
                plotMostLikelyArea(TS_Signal, Q_Signals[0], sims, TS_Signal_name, Q_Signal_name, 
                                   "Best " + SM_name + " for " + Q_Signal_name + ": " + str(sim_best), Pad, amountPeaks,
                                  toFileSave, GeneralPath + "Graphs/" + mostLikelyFile)

                if toFileSave == 0 or toFileSave == 2:
                    print("Found best peak: ", np.argmin(sims))

                    # Spacing for visability
                    print("============================================================= ")
                    print(" ")
                    print("============================================================= ")

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                if PeakOnlyData:
                    loc = peakNum
                else:
                    loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    if PeakOnlyData:
                        actualLoc  = peakToLookAt
                    else:
                        actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            ## This case do not graph.
            else:
                if i%10==0:
                    if toFileSave == 0 or toFileSave == 2:
                        if Info:
                            print("Working on Sample:", i)

                # Obtain the sliding window similarity across the time series signal given the query signals
                sims = slidingWIndow_Peaks(TS_Signal, Q_Signals, MM_Scaler, Pad, amountPeaks, typeSimM, number, nonPeakEnds,
                                          PeakOnlyData)
                SW_Sims.append(sims)

                ### Plot the most likely area in the signal sample
                sim_best = min(sims)

                # get the location of the most similar peak
                peakNum = np.argmin(sims)
                if PeakOnlyData:
                    loc = peakNum
                else:
                    loc = MapTool.getPeaksLoc(TS_Signal, Pad, peakNum)

                # get the location of the actual peaks with the query instructions
                actualLocs = []

                for peakToLookAt in peaksToLookAt:
                    if PeakOnlyData:
                        actualLoc  = peakToLookAt
                    else:
                        actualLoc = MapTool.getPeaksLoc(TS_Signal, Pad, peakToLookAt)
                    actualLocs.append(actualLoc)
                
                actualLocs = np.array(actualLocs)

                for actualLoc in actualLocs:
                    if loc == actualLoc:
                        correct = correct + 1

            count = count + 1

        SW_Sims =np.array(SW_Sims)
        
        if Info:
            printOrWriteInfo(ResultsFile, "Average Similarity: " + str(np.mean(SW_Sims)), toFileSave)

        stop = timeit.default_timer()

        if Info:
            print('Time: ', stop - start)  
        
        return SW_Sims

NameError: name 'np' is not defined

In [1]:
def corrPeakExp(Signals, Seqs, Signal_name, Seq_name, pad, peakToLookAt, corr_scale= -100, amtGraphs = 20):
    start = timeit.default_timer()
    
    count = 0
    correct =  0
    euc_corrs = []
    
    # check for errors:
    checkCount = []
    for seq in Seqs:
        checkCount.append(MapTool.getPeaks(seq, pad).shape[0])
    
    checkCount = np.array(checkCount)
    checkCount = np.unique(checkCount)
    
    if checkCount.shape[0] > 1:
        print("Error in gathering seqs as unique peaks gave different values")
        print(checkCount)
    
    
    amountPeaks = MapTool.getPeaks(Seqs[0], pad).shape[0]

    for i in range(Signals.shape[0]):

        Signal = Signals[i]

        if i < amtGraphs:
            # Title of section
            print(Signal_name + ": Signal Sample ", i)

            euc_corr = rollingEuclidean_Peaks(Signal, Seqs, corr_scale, pad, amountPeaks)
            euc_corrs.append(euc_corr)

            ### Plot the correlation
            plotCorrelation(euc_corr, "Correlation between " + Signal_name + " Sample: " + str(i) + " and " + Seq_name)

            ### Plot the most likely area in the signal sample
            corr_best = max(euc_corr)
            plotMostLikelyArea(Signal, Seqs[0], euc_corr, Signal_name, Seq_name, 
                               "Best Correlation for " + Seq_name + ", Corr is: " + str(corr_best), pad)

            print("Found best peak: ", np.argmax(euc_corr))

            # Spacing for visability
            print("============================================================= ")
            print(" ")
            print("============================================================= ")

            peakNum = np.argmax(euc_corr)
            loc = MapTool.getPeaksLoc(Signal, pad, peakNum)

            actualLocs = peakToLookAt

            actualLoc = MapTool.getPeaksLoc(Signal, pad, actualLocs)

            if loc == actualLoc:
                correct = correct + 1

        else:
            if i%10==0:
                print("Working on Sample:", i)

            ### Plot the correlation
            euc_corr = rollingEuclidean_Peaks(Signal, Seqs, corr_scale, pad, amountPeaks)
            
            euc_corrs.append(euc_corr)

            ### Plot the most likely area in the signal sample

            peakNum = np.argmax(euc_corr)
            loc = MapTool.getPeaksLoc(Signal, pad, peakNum)

            actualLocs = peakToLookAt

            actualLoc = MapTool.getPeaksLoc(Signal, pad, actualLocs)

            if loc == actualLoc:
                correct = correct + 1

        count = count + 1

    euc_corrs =np.array(euc_corrs)

    ground_truth = np.zeros((euc_corrs.shape[0], euc_corrs.shape[1]))
    ground_truth[np.arange(euc_corrs.shape[0]),peakToLookAt] = 1

    list_of_metrics = getEvaluations(euc_corrs, ground_truth, 1000)

    accumulative_metrics = ConfidenceEstimation.calculateMultipleRoundMetrics(list_of_metrics)

    auc_Score = accumulative_metrics["auc"]

    # Gather data
    fprs = [item["fall_out"] for item in list_of_metrics]
    tprs = [item["sensitivity"] for item in list_of_metrics]

    fprs = np.array(fprs)
    tprs = np.array(tprs)

    accsSec = [item["acc"] for item in list_of_metrics]

    # f1 and Acc
    ''' 
    # Other data if necessary
    f1sSec = [item["f1"] for item in list_of_metrics]
    accsSec = [item["acc"] for item in list_of_metrics]

    f1sSec = np.array(f1sSec)
    accsSec = np.array(accsSec)

    f1s.append(max(f1sSec))
    accs.append(max(accsSec))

    # truth matrix
    tnsSec = [item["tn"] for item in list_of_metrics]
    fpsSec = [item["fp"] for item in list_of_metrics]
    fnsSec = [item["fn"] for item in list_of_metrics]
    tpsSec = [item["tp"] for item in list_of_metrics]

    tnsSec = np.array(tnsSec)
    fpsSec = np.array(fpsSec)
    fnsSec = np.array(fnsSec)
    tpsSec = np.array(tpsSec)
    '''
    
    print("Results for window euclidean corralation between the Signals " + Signal_name + " and the instruction Sequences " + Seq_name)
    rocTitle = "ROC graph for all euclidean correlations."
    aucScoreSave = int(auc_Score * 1000)

    ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score)

    print("Accuracy over all samples correctly labeled: ", correct/count)
    print("Accuracy over all labels: ", max(accsSec))
    
    stop = timeit.default_timer()
    
    print('Time: ', stop - start)  

In [ ]:
'''
Tests various number of top corr comparisons for each peak correlation sequence to account for.
Produces an Accuracy for each number from 5 to 50 skiping by 5.

Does this by sum and adv. may remove sum for adv only later as this keeps in the 0-1 range for AUC gathering.
'''
def corrPeakExpTestNumCorr(Signals, Seqs, Signal_name, Seq_name, pad, peakToLookAt, corr_scale= -100, amtGraphs = 20, topNumCorr=50):
    start = timeit.default_timer()
    
    count = 0
    correct =  0
    
    # check for errors:
    checkCount = []
    for seq in Seqs:
        checkCount.append(MapTool.getPeaks(seq, pad).shape[0])
    
    checkCount = np.array(checkCount)
    checkCount = np.unique(checkCount)
    
    if checkCount.shape[0] > 1:
        print("Error in gathering seqs as unique peaks gave different values")
        print(checkCount)
    
    
    amountPeaks = MapTool.getPeaks(Seqs[0], pad).shape[0]

    for i in range(amtGraphs):

        Signal = Signals[i]

        # Title of section
        print(Signal_name + ": Signal Sample ", i)

        for number in range(0,topNumCorr, 5):
            if number == 0:
                number = 1

            print(" ")
            print("Taking the top number of correlations: ", number)

            euc_corr = rollingEuclidean_PeaksTest(Signal, Seqs, corr_scale, pad, amountPeaks, number, False)

            ### Plot the correlation
            # plotCorrelation(euc_corr, "Correlation between " + Signal_name + " Sample: " + str(i) + " and " + Seq_name)

            ### Plot the most likely area in the signal sample
            corr_best = max(euc_corr)
            # plotMostLikelyArea(Signal, Seqs[0], euc_corr, Signal_name, Seq_name, 
            #                   "Best Correlation for " + Seq_name + ", Corr is: " + str(corr_best), pad)

            print("Best Correlation: ", corr_best)
            print("Found best peak: ", np.argmax(euc_corr))

            # Accuracy by signal.
            euc_corr = np.array(euc_corr)

            ground_truth = np.zeros(euc_corr.shape)
            ground_truth[peakToLookAt] = 1

            list_of_metrics = getEvaluationsOne(euc_corr, ground_truth, 1000)

            accumulative_metrics = ConfidenceEstimation.calculateMultipleRoundMetrics(list_of_metrics)

            auc_Score = accumulative_metrics["auc"]

            # Gather data
            fprs = [item["fall_out"] for item in list_of_metrics]
            tprs = [item["sensitivity"] for item in list_of_metrics]

            fprs = np.array(fprs)
            tprs = np.array(tprs)

            accsSec = [item["acc"] for item in list_of_metrics]


            #print("Results for window euclidean corralation between the Signals " + Signal_name + " and the instruction Sequences " + Seq_name)
            #rocTitle = "ROC graph for all euclidean correlations."
            #aucScoreSave = int(auc_Score * 1000)

            #ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score)

            print("Accuracy over all labels: ", max(accsSec))

            stop = timeit.default_timer()  

In [ ]:
'''
Tests various number of top corr comparisons for each peak correlation sequence to account for.
Produces an Accuracy for each number from 5 to 50 skiping by 5.

Does this by sum and adv. may remove sum for adv only later as this keeps in the 0-1 range for AUC gathering.
'''
def corrPeakExpNumCorr(Signals, Seqs, Signal_name, Seq_name, pad, peakToLookAt, corr_scale= -100, amtGraphs = 20, topNumCorr=np.array([10])):
    
    # check for errors:
    checkCount = []
    for seq in Seqs:
        checkCount.append(MapTool.getPeaks(seq, pad).shape[0])
    
    checkCount = np.array(checkCount)
    checkCount = np.unique(checkCount)
    
    if checkCount.shape[0] > 1:
        print("Error in gathering seqs as unique peaks gave different values")
        print(checkCount)
    
    
    amountPeaks = MapTool.getPeaks(Seqs[0], pad).shape[0]
    
    for number in topNumCorr:
        start = timeit.default_timer()
    
        count = 0
        correct =  0
        euc_corrs = []
        
        
        print(" ")
        print("Taking the top number of correlations: ", number)
        
        for i in range(Signals.shape[0]):

            Signal = Signals[i]

            if i < amtGraphs:
                # Title of section
                print(Signal_name + ": Signal Sample ", i)

                euc_corr = rollingEuclidean_Peaks(Signal, Seqs, corr_scale, pad, amountPeaks, number)
                euc_corrs.append(euc_corr)

                ### Plot the correlation
                plotCorrelation(euc_corr, "Correlation between " + Signal_name + " Sample: " + str(i) + " and " + Seq_name)

                ### Plot the most likely area in the signal sample
                corr_best = max(euc_corr)
                plotMostLikelyArea(Signal, Seqs[0], euc_corr, Signal_name, Seq_name, 
                                   "Best Correlation for " + Seq_name + ", Corr is: " + str(corr_best), pad)

                print("Found best peak: ", np.argmax(euc_corr))

                # Spacing for visability
                print("============================================================= ")
                print(" ")
                print("============================================================= ")

                peakNum = np.argmax(euc_corr)
                loc = MapTool.getPeaksLoc(Signal, pad, peakNum)

                actualLocs = peakToLookAt

                actualLoc = MapTool.getPeaksLoc(Signal, pad, actualLocs)

                if loc == actualLoc:
                    correct = correct + 1

            else:
                if i%10==0:
                    print("Working on Sample:", i)

                ### Plot the correlation
                euc_corr = rollingEuclidean_Peaks(Signal, Seqs, corr_scale, pad, amountPeaks, number)

                euc_corrs.append(euc_corr)

                ### Plot the most likely area in the signal sample

                peakNum = np.argmax(euc_corr)
                loc = MapTool.getPeaksLoc(Signal, pad, peakNum)

                actualLocs = peakToLookAt

                actualLoc = MapTool.getPeaksLoc(Signal, pad, actualLocs)

                if loc == actualLoc:
                    correct = correct + 1

            count = count + 1

        euc_corrs =np.array(euc_corrs)

        ground_truth = np.zeros((euc_corrs.shape[0], euc_corrs.shape[1]))
        ground_truth[np.arange(euc_corrs.shape[0]),peakToLookAt] = 1

        list_of_metrics = getEvaluations(euc_corrs, ground_truth, 1000)

        accumulative_metrics = ConfidenceEstimation.calculateMultipleRoundMetrics(list_of_metrics)

        auc_Score = accumulative_metrics["auc"]

        # Gather data
        fprs = [item["fall_out"] for item in list_of_metrics]
        tprs = [item["sensitivity"] for item in list_of_metrics]

        fprs = np.array(fprs)
        tprs = np.array(tprs)

        accsSec = [item["acc"] for item in list_of_metrics]

        print("Results for window euclidean corralation between the Signals " + Signal_name + " and the instruction Sequences " + Seq_name)
        rocTitle = "ROC graph for all euclidean correlations."
        aucScoreSave = int(auc_Score * 1000)

        ROC.plotROC(fprs, tprs, "red", 10, "ROC", "FP Rate", "TP Rate", rocTitle, 4, auc_Score)

        print("Accuracy over all samples correctly labeled: ", correct/count)
        print("Accuracy over all labels: ", max(accsSec))

        stop = timeit.default_timer()

        print('Time: ', stop - start)  